# Fair-Seldonian on real data: UCI Adult income

This notebook applies the Quasi-Seldonian Algorithm (QSA) to the UCI Adult
income dataset under a **demographic-parity** constraint, and shows both sides of
the Seldonian guarantee on real data:

* an ordinary logistic regression reaches good accuracy but predicts the positive
  class at very different rates for the two groups (a demographic-parity gap), and
* QSA, asked to certify that gap is bounded, returns **No Solution Found**
  rather than shipping the biased model.

**Demographic parity** asks that the predicted-positive rate `P(pred = 1)` be
close across groups, regardless of the true label.

**Task framing**

| symbol | meaning |
|--------|---------|
| `Y` (label) | income > 50K |
| `T` (sensitive) | sex (1 = Male, 0 = Female) |
| `X` (features) | standardized numeric columns, with `T` appended as the final column (library convention) |

> Requires network access on the first run to download the dataset (cached afterwards).

In [1]:
import numpy as np
from sklearn.datasets import fetch_openml

from fair_seldonian.algorithms import QSA
from fair_seldonian.config import SeldonianConfig
from fair_seldonian.models import predict, simple_logistic

## 1. Load and frame the data

We download Adult, derive the binary label and sensitive attribute, standardize
the numeric features, and append the sensitive attribute as the final feature
column. Then we take a deterministic subsample and train/test split.

In [2]:
frame = fetch_openml("adult", version=2, as_frame=True, parser="auto").frame.dropna()

T = (frame["sex"].astype(str) == "Male").astype(int).to_numpy()
Y = frame["class"].astype(str).str.contains(">50K").astype(int).to_numpy()

numeric = frame.select_dtypes("number")
standardized = (numeric - numeric.mean()) / numeric.std()
X = np.column_stack([standardized.to_numpy(), T]).astype(float)

# deterministic subsample + split for a fast, reproducible demo
rng = np.random.default_rng(0)
idx = rng.permutation(len(X))[:8000]
X, Y, T = X[idx], Y[idx], T[idx]
cut = int(0.7 * len(X))
X_tr, Y_tr, T_tr = X[:cut], Y[:cut], T[:cut]
X_te, Y_te, T_te = X[cut:], Y[cut:], T[cut:]

print(
    f"Adult: {len(X)} examples, positive rate {Y.mean():.3f}, male share {T.mean():.3f}"
)

Adult: 8000 examples, positive rate 0.241, male share 0.678


## 2. Unconstrained baseline

A plain logistic regression. We measure overall accuracy and the
predicted-positive rate within each group; the gap between those rates is the
demographic-parity violation the fairness constraint targets.

In [3]:
def positive_rate(pred, mask):
    # P(pred = 1 | T = group): the predicted-positive rate for a group
    return float(pred[mask].mean()) if mask.any() else float("nan")


theta, theta1 = simple_logistic(X_tr, Y_tr)
pred = (predict(theta, theta1, X_te).detach().numpy() >= 0.5).astype(int)

acc = float((pred == Y_te).mean())
pr_male = positive_rate(pred, T_te == 1)
pr_female = positive_rate(pred, T_te == 0)
dp_gap = abs(pr_male - pr_female)

print(f"accuracy               : {acc:.3f}")
print(f"positive rate (male)   : {pr_male:.3f}")
print(f"positive rate (female) : {pr_female:.3f}")
print(f"demographic-parity gap : {dp_gap:.3f}")

accuracy               : 0.814
positive rate (male)   : 0.196
positive rate (female) : 0.045
demographic-parity gap : 0.151


## 3. The Seldonian guarantee (demographic parity)

Now we ask QSA to return a model only if it can certify the demographic-parity
constraint holds with high probability. On this data it declines.

In [4]:
# Demographic parity: equal predicted-positive rate across groups.
# Within a group, P(pred = 1) = TP + FP, so the constraint (in postfix form) is
# |(TP(1) + FP(1)) - (TP(0) + FP(0))| - 0.10 <= 0.
constraint = "TP(1) FP(1) + TP(0) FP(0) + - abs 0.1 -"
config = SeldonianConfig(constraint=constraint)

_, _, passed = QSA(X_tr, Y_tr, T_tr, "opt", None, None, config)

if passed:
    print("certified: demographic parity holds with high confidence")
else:
    print("No Solution Found - QSA will not certify demographic parity here,")
    print("rather than return a model with the disparity shown above.")

No Solution Found - QSA will not certify demographic parity here,
rather than return a model with the disparity shown above.


## Takeaway

The unconstrained model is accurate but predicts high income far more often for
one group than the other. QSA trades coverage for safety: on data where it
cannot *prove* demographic parity holds, it returns no model at all - never an
unsafe one. See [`quickstart.ipynb`](quickstart.ipynb) for cases where QSA
*does* certify, and [`custom_constraint.py`](custom_constraint.py) for other
fairness definitions.